In [1]:
import pandas as pd

In [13]:
df = pd.read_csv("../Datasets/npl_dataset.csv")
df.sample(5)

,Unnamed: 0,match_id,inning,batting_team,bowling_team,ball_over,ball_result,bowler,batsman,non_striker,...,player_dismissed,dismissal_kind,fielder,batsman_runs,wide_runs,bye_runs,legbye_runs,noball_runs,extra_runs,total_runs
6170,6170,2024_27,2,Pokhara Avengers,Sudurpaschim Royals,0.3,0,Harmeet Singh,Dinesh Kharel,Andries Gous,...,NaN,NaN,NaN,0,0,0,0,0,0,0
14360,14360,2025_30,1,Kathmandu Gorkhas,Lumbini Lions,12.2,1,Tilak Bhandari,Santosh Yadav,Mohammad Aadil Alam,...,NaN,NaN,NaN,1,0,0,0,0,0,1
2426,2426,2024_11,1,Lumbini Lions,Pokhara Avengers,18.1,1w,Raymon Reifer,Tom Moores,Rohit Paudel,...,NaN,NaN,NaN,0,1,0,0,0,1,1
6681,6681,2024_29,2,Chitwan Rhinos,Karnali Yaks,6.6,1,William Bosisto,Kushal Malla,Ravi Bopara,...,NaN,NaN,NaN,1,0,0,0,0,0,1
10675,10675,2025_14,2,Janakpur Bolts,Pokhara Avengers,4.2,0,Bipin Khatri,Sanjay Krishnamurthi,Anil Sah,...,NaN,NaN,NaN,0,0,0,0,0,0,0


In [3]:
#For most runs
most_runs = df.groupby("batsman")["batsman_runs"].sum().sort_values(ascending=False).head()
most_runs

batsman
Rohit Paudel            555
Ravi Bopara             523
Binod Bhandari          499
Saif Zaib               391
Dipendra Singh Airee    389
Name: batsman_runs, dtype: int64

In [4]:
#Most Sixes
mask = df["batsman_runs"] == 6
new_df = df[mask]
new_df["batsman"].value_counts().head()

batsman
James Neesham           31
Ravi Bopara             24
Lokesh Bam              23
Dipendra Singh Airee    21
Kushal Malla            21
Name: count, dtype: int64

In [5]:
#Most Fours
mask = df["batsman_runs"] == 4
new_df = df[mask]
new_df["batsman"].value_counts().head()

batsman
Rohit Paudel       44
Binod Bhandari     42
Lahiru Milantha    38
Ravi Bopara        36
Adam Rossington    31
Name: count, dtype: int64

In [6]:
#Highest Scores
result = (
        df.groupby(["match_id", "batsman"])["batsman_runs"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .drop_duplicates(subset="match_id", keep="first")
        [["batsman", "batsman_runs"]]
        .head(5)
)
result

,batsman,batsman_runs
0,Mark Watt,114
1,Adam Rossington,108
2,Andries Gous,104
4,Saif Zaib,90
5,Priyank Panchal,90


In [7]:
#Most Hundreds
total_runs = df.groupby(["match_id" , "batsman"])["batsman_runs"].sum().reset_index(name="runs")
hundreds = total_runs[total_runs["runs"]>100]
hundreds["batsman"].value_counts().reset_index()

,batsman,count
0,Andries Gous,1
1,Adam Rossington,1
2,Mark Watt,1


In [8]:
#Most Fifties
total_runs = df.groupby(["match_id" , "batsman"])["batsman_runs"].sum().reset_index(name="runs")
fifties = total_runs[(total_runs["runs"]>50) & (total_runs["runs"]<100)]
fifties["batsman"].value_counts().reset_index()

,batsman,count
0,Ravi Bopara,6
1,Binod Bhandari,3
2,Lahiru Milantha,2
3,Rohit Paudel,2
4,Martin Guptill,2
5,William Bosisto,2
6,Anil Sah,2
7,Basir Ahamad,2
8,Saif Zaib,2
9,Lokesh Bam,2


In [155]:
#Best Batting Strike Rate

#Calculate Total Runs
total_runs = df.groupby(["batsman"])["batsman_runs"].sum().reset_index(name = "runs")

#Calculate Total Balls
legal_balls = df[df["wide_runs"]==0] #exclude wide
total_balls = legal_balls.groupby("batsman").size().reset_index(name="balls")
eligible_batsman = total_balls[total_balls["balls"] > 10] #mimium 10 balls played

#Calculate StrikeRate
combined = total_runs.merge(eligible_batsman , on="batsman")
combined["strike_rate"] = (combined["runs"] / combined["balls"])*100
result = combined[["batsman" , "strike_rate"]].reset_index().sort_values(by="strike_rate" , ascending=False)
result

,index,batsman,strike_rate
34,34,Dan Douthwaite,187.179487
61,61,James Neesham,185.026738
93,93,Nar Sarki,183.333333
111,111,Rishi Dhawan,174.193548
65,65,John Simpson,166.990291
...,...,...,...
71,71,Kishore Mahato,48.148148
137,137,Subash Bhandari,46.153846
106,106,Ramon Simmonds,46.153846
8,8,Abinash Bohara,41.666667


In [152]:
#Best Batting Average

#Calculating Total Batsman
total_runs = df.groupby(["batsman"])["batsman_runs"].sum().reset_index(name="runs")

#Calculating Total Innings Played
innings_played = (
    df[["match_id", "inning", "batsman"]]
      .drop_duplicates()
      .groupby("batsman")
      .size()
      .reset_index(name="innings")
)

#Calculating Total dismissed innings
innings_out_df = df[(df["player_dismissed"].notna()) & (df["player_dismissed"] == df["batsman"])]
innings_outs = innings_out_df[["match_id" , "batsman" , "player_dismissed"]].groupby("batsman").size().reset_index(name="outs")

#Merging InningPlayed and InningDismissed
innings_df = innings_outs.merge(innings_played ,   on="batsman")

#Combined all 3 dataframes
combined_df = innings_df.merge(total_runs ,   on="batsman")
#Calculating Not Outs
combined_df["not_outs"] = combined_df["innings"] - combined_df["outs"]
#Calculating Average
combined_df["average"] = round(combined_df["runs"]/(combined_df["innings"] - combined_df["not_outs"]) , 1)
#Making Final Results
result = combined_df[["batsman" , "average"]].reset_index().sort_values(by="average" , ascending=False).head(10)
result

,index,batsman,average
154,154,Wayne Parnell,64.0
116,116,Rishi Dhawan,54.0
11,11,Adam Rossington,53.8
43,43,Dhananjaya Lakshan,53.0
27,27,Bibek Yadav,50.0
138,138,Shikhar Dhawan,45.3
14,14,Andries Gous,44.8
155,155,William Bosisto,41.4
85,85,Mark Watt,41.0
56,56,Gulshan Jha,40.3
